In [ ]:
import torch
import pandas as pd

from google.colab import files
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Lettura dataset

In [ ]:
uploaded = files.upload()
df = pd.read_excel(list(uploaded.keys())[0])

# Rimuove le ultime due colonne
df = df.iloc[:, :-2]

# Crea dizionario: termine -> lista definizioni
term_dict = {
    row[1]: [x for x in row[2:] if pd.notna(x)]
    for row in df.values
}

Saving testi_tradotti.xlsx to testi_tradotti.xlsx


IndexError: index 1 is out of bounds for axis 0 with size 1

# Caricamento del Modello

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
  "microsoft/Phi-3.5-mini-instruct",
  device_map="auto",
  torch_dtype="auto",
  trust_remote_code=False
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/2.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/195 [00:00<?, ?B/s]

# Caricamento del Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
  "microsoft/Phi-3.5-mini-instruct",
  trust_remote_code=True
)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

#Creazione della pipeline di generazione

In [ ]:
pipe = pipeline(
  "text-generation",
  model=model,
  tokenizer=tokenizer,
  max_new_tokens=300,
  temperature=0.2,
  top_p=1
)

Device set to use cuda:0


# Definizione del prompt per le definizionio del lab-2

In [ ]:
prompt_list = []
for _, defs in term_dict.items():
  msg = [
        {
          "role": "user",
          "content": (
              "Stai svolgendo un esercizio di labeling: ti verrà fornita una lista di definizioni e dovrai indovinare la parola corrispondente."
              "Rispondi solo con la parola che meglio corrisponde alla lista di definizioni. Non aggiungere spiegazioni."
              f"Definizioni :\n" + "\n".join(defs) + "\n\n"
          )
        }
      ]
  prompt_list.append(msg)


['Indumento per la parte inferiore del corpo umano', 'Indumento per le gambe, diviso per ogni gamba può essere lungo o corto e di diversi tessuti', 'abito indossato sulle gambe', 'capo di abbigliamento per le gambe', 'Indumento indossato nella parte inferiore del corpo, copre le gambe.', 'Indumento per la parte inferiore del corpo che può coprire parzialmente o totalmente le gambe.', 'Capo indossabile che copre la parte inferiore del corpo, in cui ogni gamba è coperta singolarmente', 'indumento indossabile da una persona tipicamente di colore blu e particolarmente resistente', 'Indumento che copre le gambe di una persona', 'Indumento da indossare dalla vita in giù', 'Indumento vestibile nella parte bassa del corpo', 'Indumento di diverso materiale che si indossa per coprire le gambe', 'capo di abbigliamento per le gambe, possono essere di diverso materiale', 'Indumento per coprire le gambe', 'vestiario da indossare sulle gambe', 'indumento indossabile nella parte inferiore del corpo', 

# Generazione del risultato per le definizioni


In [ ]:
output_list = []
for msg in prompt_list:
  output = pipe(msg)
  output_list.append(output[0]["generated_text"][1]["content"])

print(output_list)

[' pantaloni', ' microscopio', ' rischio', ' costo di ottimizzazione']


#Definizione del prompt per i topic del lab-4

In [ ]:
prompt_list = []
topic_list = [
    "0_militare_dell_guerra_militari",
    "1_web_dati_sviluppo_gestione",
    "2_social_linguaggio_modelli_parole",
    "3_immagini_reti_dati_neurali",
    "4_virtuale_realtà_3d_dell",
    "5_applicazione_mobile_un_android",
    "6_sicurezza_attacchi_cyber_vulnerabilitã",
    "7_cloud_servizi_infrastruttura_computing"
]

for topic in topic_list:
  msg = [
        {
          "role": "user",
          "content": (
              "Stai svolgendo un esercizio di labeling dei topic. Ti verrà fornita una lista di parole chiave relative a ciascun topic."
              "Il tuo compito è di generare una singola etichetta breve e chiara che descriva il tema principale del topic."
              "Rispondi con una sola frase o una parola sintetica che catturi il significato principale del topic."
              "Non aggiungere spiegazioni."
              f"Topic da etichettare :\n" + topic + "\n\n"
          )
        }
      ]
  prompt_list.append(msg)

# Generazione del risultato per i topic

In [ ]:
output_list = []
for msg in prompt_list:
  output = pipe(msg)
  output_list.append(output[0]["generated_text"][1]["content"])

print(output_list)

[' Guerra Militare', ' Sviluppo e gestione dei dati web', ' Modelli linguistici sociali\n\n##### Nota:\nLa frase "Modelli linguistici sociali" è una sintesi che combina i concetti di socialità, linguaggio e modelli, che sono centrali per il tema dato. Evita dettagli e mantiene la chiarezza, come richiesto.', ' Neurali Immagini Reti\n', ' VR 3D Dell', ' App Android Mobile', ' Sicurezza Cyber e Vulnerabilità', ' Cloud Infrastructure Computing']


[
  0' Guerra Militare',
  1' Sviluppo e gestione dei dati web',
  2' Modelli linguistici sociali\n\n
  3' Neurali Immagini Reti\n',
  4' VR 3D Dell',
  5' App Android Mobile',
  6' Sicurezza Cyber e Vulnerabilità',
  7' Cloud Infrastructure Computing'
]
